# Reproject, resample & align

Three ways to change a raster's grid:

- **`to_crs(epsg)`** — reproject to another coordinate reference system.
- **`resample(cell_size)`** — change the pixel size (resolution) within the same CRS.
- **`align(reference)`** — snap rows/columns/cell-size/CRS to match another raster. This is
  the operation you use to make every layer line up with a DEM before modelling.

## Setup

In [1]:
import os

os.environ['MPLBACKEND'] = 'Agg'  # never trigger an interactive backend

import tempfile
from pathlib import Path

import numpy as np


def _find_data():
    for base in [Path.cwd(), *Path.cwd().parents]:
        cand = base / 'tests' / 'data'
        if cand.is_dir():
            return cand.resolve()
    raise FileNotFoundError('Could not locate tests/data from ' + str(Path.cwd()))


DATA = _find_data()
WORK = Path(tempfile.mkdtemp(prefix='pyramids-ops-'))
DATA.is_dir(), WORK.is_dir()

(True, True)

In [2]:
from pyramids.dataset import Dataset

ds = Dataset.read_file(str(DATA / 'acc4000.tif'))
ds.shape, ds.epsg, ds.cell_size

2026-06-08 23:01:46 | INFO | pyramids.base.config | Logging is configured.


((1, 13, 14), 32618, 4000.0)

## Reproject — `to_crs`

Reproject from UTM (EPSG:32618) to geographic lon/lat (EPSG:4326).

In [3]:
wgs84 = ds.to_crs(4326)
wgs84.epsg, wgs84.shape, wgs84.cell_size

(4326, (1, 13, 14), 0.03611587177268461)

## Resample — `resample`

Double the cell size; the row/column count roughly halves.

In [4]:
coarser = ds.resample(cell_size=ds.cell_size * 2)
coarser.cell_size, coarser.shape

(8000.0, (1, 6, 7))

## Align to a reference — `align`

`align` copies the reference raster's grid (rows, columns, cell size, CRS) onto the data, so
the two share an identical geotransform — the prerequisite for cell-by-cell raster algebra.

In [5]:
reference = ds.resample(cell_size=ds.cell_size * 2)  # stand-in reference grid
aligned = ds.align(reference)
aligned.shape == reference.shape, aligned.cell_size == reference.cell_size

(True, True)

## Notes

- `to_crs` / `resample` take a `method=` (`"nearest neighbor"`, `"bilinear"`, `"cubic"`);
  use nearest for categorical data, bilinear/cubic for continuous.
- See also: [Crop & mask](crop-mask.ipynb), [Mosaic / merge](mosaic-merge.ipynb).